# Put together ground truth images

In [1]:
# libraries
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
#from matplotlib.colors import ListedColormap
#from ipywidgets import interact, fixed
#from IPython.display import clear_output
import SimpleITK as sitk
import glob
import re
from collections import defaultdict

## Ground truths will be placed with mha files, with title combined_gt_stack*.mha

In [5]:
label_map = {
    "_L_GR": 1,
    "_L_HS": 2,
    "_L_QF": 3,
    "_L_SA": 4,
    "_R_GR": 5,
    "_R_HS": 6,
    "_R_QF": 7,
    "_R_SA": 8
}

folders = glob.glob("myosegmenTUM/*/SegmentationMasks")

all_outputs = []

for folder in folders:
    print(folder)
    files = glob.glob(os.path.join(folder, "*.mha"))

    # group files by stack
    stacks = defaultdict(list)

    for f in files:
        filename = os.path.basename(f)

        match = re.search(r"(stack\d+)", filename)
        if match:
            stack_id = match.group(1)
            stacks[stack_id].append(f)

    # process each stack separately
    for stack_id, stack_files in stacks.items():

        combined_gt = None
        reference_img = None

        for f in stack_files:
            filename = os.path.basename(f)

            for key, label in label_map.items():
                if key in filename:
                    img = sitk.ReadImage(f)
                    arr = sitk.GetArrayFromImage(img)

                    if combined_gt is None:
                        combined_gt = np.zeros_like(arr)
                        reference_img = img

                    combined_gt[arr > 0] = label
                    break

        # convert back
        out = sitk.GetImageFromArray(combined_gt)
        out.CopyInformation(reference_img)

        # save per stack
        output_path = os.path.join(folder, f"combined_gt_{stack_id}.mha")
        sitk.WriteImage(out, output_path)

        all_outputs.append(out)

print(f"Processed {len(all_outputs)} stacks.")

myosegmenTUM\HV001_1\SegmentationMasks
myosegmenTUM\HV001_2\SegmentationMasks
myosegmenTUM\HV001_3\SegmentationMasks
myosegmenTUM\HV002_1\SegmentationMasks
myosegmenTUM\HV002_2\SegmentationMasks
myosegmenTUM\HV002_3\SegmentationMasks
myosegmenTUM\HV003_1\SegmentationMasks
myosegmenTUM\HV003_2\SegmentationMasks
myosegmenTUM\HV003_3\SegmentationMasks
myosegmenTUM\HV004_1\SegmentationMasks
myosegmenTUM\HV005_1\SegmentationMasks
myosegmenTUM\HV006_1\SegmentationMasks
myosegmenTUM\HV007_1\SegmentationMasks
myosegmenTUM\HV008_1\SegmentationMasks
myosegmenTUM\HV009_1\SegmentationMasks
myosegmenTUM\HV010_1\SegmentationMasks
myosegmenTUM\HV011_1\SegmentationMasks
myosegmenTUM\HV012_1\SegmentationMasks
myosegmenTUM\HV013_1\SegmentationMasks
myosegmenTUM\HV014_1\SegmentationMasks
myosegmenTUM\HV015_1\SegmentationMasks
myosegmenTUM\P001_1\SegmentationMasks
myosegmenTUM\P002_1\SegmentationMasks
myosegmenTUM\P003_1\SegmentationMasks
myosegmenTUM\P004_1\SegmentationMasks
Processed 54 stacks.


### view example

In [ ]:
# just a slice

In [ ]:
example = 'myosegmenTUM/HV003_1/SegmentationMasks/combined_gt_stack1.mha'
example = sitk.ReadImage(example)
example = sitk.GetArrayFromImage(example)
plt.imshow(example[10,:,:])